In [27]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, split, explode
from pyspark.sql import Window
from pyspark.sql.functions import dense_rank,row_number,monotonically_increasing_id
from pyspark.sql.functions import broadcast

In [28]:
def build_spark_session():
    spark = SparkSession.builder \
    .appName("IMDb Lakehouse Pipeline ClickHouse Load") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.warehouse.dir", "s3a://imdb-bucket/hive_catalog/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "lakehouse_admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "lakehouse_password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")\
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")\
    .config("spark.executor.cores", "2") \
    .config("spark.cores.max", "4") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.parquet.compression.codec", "snappy")\
    .enableHiveSupport()\
    .config("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog") \
    .config("spark.sql.catalog.clickhouse.host", "clickhouse-olap") \
    .config("spark.sql.catalog.clickhouse.http_port", "8123") \
    .config("spark.sql.catalog.clickhouse.user", "clickhouse_admin") \
    .config("spark.sql.catalog.clickhouse.password", "clickhouse_password") \
    .config("spark.sql.catalog.clickhouse.database", "imdb_olap")\
    .config("spark.clickhouse.write.format", "json")\
    .getOrCreate()
    
    return spark

In [30]:
spark = build_spark_session()

26/08/25 04:33:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [19]:
spark.sql("USE imdb_olap_gold")
title_principals = spark.sql("SELECT * FROM title_principals")
dim_genres = spark.sql("SELECT * FROM dim_genres")
dim_peoples = spark.sql("SELECT * FROM dim_peoples")
dim_professions = spark.sql("SELECT * FROM dim_professions")
dim_titles = spark.sql("SELECT * FROM dim_titles")
fact_people_related = spark.sql("SELECT * FROM fact_people_related")
fact_title_related = spark.sql("SELECT * FROM fact_title_related")
fact_final= spark.sql("SELECT * FROM fact_final")
spark.sql('show tables in imdb_olap_gold;').show()

+--------------+-------------------+-----------+
|     namespace|          tableName|isTemporary|
+--------------+-------------------+-----------+
|imdb_olap_gold|   title_principals|      false|
|imdb_olap_gold|         dim_genres|      false|
|imdb_olap_gold|        dim_peoples|      false|
|imdb_olap_gold|    dim_professions|      false|
|imdb_olap_gold|fact_people_related|      false|
|imdb_olap_gold| fact_title_related|      false|
|imdb_olap_gold|         dim_titles|      false|
|imdb_olap_gold|         fact_final|      false|
|imdb_olap_gold|         title_akas|      false|
+--------------+-------------------+-----------+



In [5]:
dim_genres.printSchema()
spark.sql(""" 
CREATE OR REPLACE TABLE clickhouse.imdb_olap.dim_genres(
    genre String,
    genre_id INT
)USING ClickHouse
TBLPROPERTIES (
  engine = 'MergeTree()',
  order_by = 'genre_id',
  'settings.allow_nullable_key' = '1',
  'settings.index_granularity' = '8192'
);
""")
dim_genres.writeTo("clickhouse.imdb_olap.dim_genres").append()

root
 |-- genre: string (nullable = true)
 |-- genre_id: integer (nullable = true)



In [10]:
dim_peoples.printSchema()
spark.sql(""" 
CREATE OR REPLACE TABLE clickhouse.imdb_olap.dim_peoples(
    nconst String,
    primaryName String,
    birthYear INT,
    deathYear INT
)USING ClickHouse
TBLPROPERTIES (
  engine = 'MergeTree()',
  order_by = 'birthYear',
  'settings.allow_nullable_key' = '1',
  'settings.index_granularity' = '8192'
);
""")
dim_peoples.writeTo("clickhouse.imdb_olap.dim_peoples").append()

root
 |-- nconst: string (nullable = true)
 |-- primaryName: string (nullable = true)
 |-- birthYear: integer (nullable = true)
 |-- deathYear: integer (nullable = true)



In [12]:
dim_professions.printSchema()
spark.sql(""" 
CREATE OR REPLACE TABLE clickhouse.imdb_olap.dim_professions(
    profession String,
    profession_id INT
)USING ClickHouse
TBLPROPERTIES (
  engine = 'MergeTree()',
  order_by = 'profession_id',
  'settings.allow_nullable_key' = '1',
  'settings.index_granularity' = '8192'
);
""")
dim_professions.writeTo("clickhouse.imdb_olap.dim_professions").append()

root
 |-- profession: string (nullable = true)
 |-- profession_id: integer (nullable = true)



In [17]:
dim_titles.printSchema()
spark.sql(""" 
CREATE OR REPLACE TABLE clickhouse.imdb_olap.dim_titles(
    tconst String,
    titleType String,
    originalTitle String,
    isAdult INT,
    startYear INT,
    endYear INT,
    runtimeMinutes String
)USING ClickHouse
TBLPROPERTIES (
  engine = 'MergeTree()',
  PARTITION = 'titleType,startYear',
  order_by = 'titleType,startYear',
  'settings.allow_nullable_key' = '1',
  'settings.index_granularity' = '8192'
);
""")
dim_titles.writeTo("clickhouse.imdb_olap.dim_titles").append()

root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: string (nullable = true)



In [23]:
fact_final.printSchema()
spark.sql(""" 
CREATE OR REPLACE TABLE clickhouse.imdb_olap.fact_final(
    tconst String,
    nconst String,
    profession_id Int,
    genre_id Int,
    averageRating Float,
    numVotes Int,
    primaryTitle String,
    originalTitle String,
    isAdult Int,
    endYear Int,
    runtimeMinutes Int,
    titleType String,
    startYear Int
)USING ClickHouse
TBLPROPERTIES (
  engine = 'MergeTree()',
  PARTITION = 'titleType,startYear',
  order_by = 'titleType,startYear',
  'settings.allow_nullable_key' = '1',
  'settings.index_granularity' = '8192'
);
""")
fact_final.writeTo("clickhouse.imdb_olap.fact_final").append()

root
 |-- tconst: string (nullable = true)
 |-- nconst: string (nullable = true)
 |-- profession_id: integer (nullable = true)
 |-- genre_id: integer (nullable = true)
 |-- averageRating: float (nullable = true)
 |-- numVotes: integer (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- startYear: integer (nullable = true)



In [32]:
print("ClickHouse LOAD Pipeline Execution Completed Successfully.")
spark.stop()

ClickHouse LOAD Pipeline Execution Completed Successfully.
